# Dow 30 joint allocation extension

## Research question

How does a single PPO policy allocate across all 30 fixed Dow constituents and cash, and how do its initial allocation, subsequent trading, and exposure-composition paths compare within that universe?

In [ ]:
from pathlib import Path
from collections import deque
import json
import re
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (START, *START.parents) if (candidate / "pyproject.toml").is_file()),
    None,
)
assert ROOT is not None, "Could not locate the project root containing pyproject.toml."
print(f"Python: {sys.version.split()[0]}")
print(f"Project: {ROOT}")

## Inputs and provenance

This extension uses all 30 constituents in the existing processed snapshot through 2026-08-31. It has its own frozen training-only scaler, run directory, and report. The primary three-stock experiment and its 18 trained models are not altered. The fixed 2026-08-31 membership backfills historical dates and therefore has survivorship bias.

In [ ]:
from dtasrl.config import load_config
from dtasrl.provenance import run_directory
from dtasrl.storage import verify_inventory
from dtasrl.data.inputs import load_real_inputs
import sys

sys.path.insert(0, str(ROOT))
from scripts.dow30_extension import CONFIG, OUTPUT, SEEDS, prepare_inputs, report

cfg = load_config(CONFIG)
frozen = prepare_inputs()
verify_inventory(frozen)
prepared, data_hash = load_real_inputs(cfg)
print(f"Frozen input: {frozen}")
print(f"Processed data SHA-256: {data_hash}")
print(f"Tradable stocks: {len(cfg['tickers'])}; actor actions: {len(cfg['tickers']) + 1}")

## Data checks

In [ ]:
assert len(cfg["tickers"]) == len(cfg["observation_tickers"]) == 30
assert cfg["tickers"] == prepared["train"]["tickers"]
assert cfg["tickers"] == prepared["test"]["tickers"]
rows = []
for name in ("train", "validation", "test"):
    data = prepared[name]
    assert data["prices"].shape[1] == 30
    assert data["features"].shape[1] == 240
    rows.append({"split": name, "first observation": str(data["dates"][0])[:10],
                 "last observation": str(data["dates"][-1])[:10],
                 "rows including prior-close boundary": len(data["dates"]),
                 "stocks": data["prices"].shape[1], "features": data["features"].shape[1]})
display(pd.DataFrame(rows))

## Configuration

In [ ]:
from dtasrl.config import public_config
display(pd.DataFrame([{"parameter": key, "value": str(public_config(cfg)[key])}
                      for key in ("algorithm", "backend", "total_timesteps", "transaction_cost_bps",
                                  "execution", "stage", "output_group")]))
print("Seeds displayed in the paper as 1, 2, 3:", dict(zip((1, 2, 3), SEEDS)))

## Execute experiments

The default is report-only. Set `RUN_EXPERIMENTS = True` to train three independent 5,001,216-transition models manually. Runs execute sequentially. A completed run is skipped. An incomplete run resumes only when a complete checkpoint exists; otherwise the notebook stops for inspection. No existing three-stock model or output is changed.

In [ ]:
RUN_EXPERIMENTS = False
execution_log = []
if RUN_EXPERIMENTS:
    from collections import deque
    import re
    import subprocess
    for seed in SEEDS:
        run_path = run_directory(cfg, seed)
        complete = (run_path / "manifest.json").is_file() and (run_path / "metrics.json").is_file()
        if complete:
            verify_inventory(run_path)
            print(f"SKIP complete | Dow 30 | seed {SEEDS.index(seed) + 1}")
            execution_log.append({"seed": seed, "action": "skipped_complete", "run": str(run_path)})
            continue
        command = [sys.executable, "-u", "-m", "dtasrl.cli", "experiment", "run",
                   "--config", str(CONFIG.relative_to(ROOT)), "--seed", str(seed)]
        if run_path.exists():
            pointer = run_path / "latest_checkpoint.json"
            if not ((run_path / "training_identity.json").is_file() and pointer.is_file()):
                raise RuntimeError(f"Incomplete run has no safe checkpoint: {run_path}")
            checkpoint = json.loads(pointer.read_text(encoding="utf-8"))
            folder = run_path / checkpoint["path"]
            if not all((folder / name).is_file() for name in ("checkpoint.json", "model.zip", "state.pkl")):
                raise RuntimeError(f"Checkpoint is incomplete: {folder}")
            command.append("--resume")
            action = "resumed"
        else:
            action = "started"
        print(f"{action.upper()} | Dow 30 | seed {SEEDS.index(seed) + 1}", flush=True)
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1)
        assert process.stdout is not None
        display_handle = display("Starting progress bar...", display_id=True)
        recent = deque(maxlen=80)
        ansi = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")
        frame = []
        while character := process.stdout.read(1):
            if character in ("\r", "\n"):
                rendered = ansi.sub("", "".join(frame)).strip()
                frame.clear()
                if rendered:
                    recent.append(rendered)
                    display_handle.update(rendered)
            else:
                frame.append(character)
        if frame:
            recent.append(ansi.sub("", "".join(frame)).strip())
        return_code = process.wait()
        if return_code:
            print("\n".join(recent))
            raise subprocess.CalledProcessError(return_code, command)
        display_handle.update(f"COMPLETED | Dow 30 | seed {SEEDS.index(seed) + 1}")
        execution_log.append({"seed": seed, "action": action, "run": str(run_path)})
display(pd.DataFrame(execution_log))

## Results

In [ ]:
extension = report()
display(extension["status"])
if extension["summary"].empty:
    print("Formal Dow 30 results: not_started. No performance or attribution claim is available.")
else:
    display(extension["summary"])
    display(extension["effects"])

## Figures

In [ ]:
for path in extension["figures"]:
    display(Image(filename=str(path)))
print(f"Saved {len(extension['figures'])} PNG figure(s) under {OUTPUT / 'figures'}.")

## Interpretation and limitations

Within the fixed 30-stock universe, Full PPO is compared with equal-weight buy-and-hold, Freeze-1/5/20/63, and the pre-specified 2×2 exposure-composition paths. These are pathwise accounting comparisons, not proof of market predictability. The three-stock and 30-stock policies change both the opportunity set and observation/action dimensions; differences between them cannot be attributed exclusively to the number of stocks. The 2026 membership backfill is not a historically investable Dow strategy. A three-seed extension is descriptive and is not added to the primary 18-run inference.

## Output inventory

In [ ]:
display(pd.read_csv(OUTPUT / "inventory.csv"))
print("Every completed run retains its model, rollout metrics, full evaluation trajectory, actions, costs, counterfactual paths and checksums.")